# 05 - Hasil val: tabel 16 run dan kunci konfigurasi

Alur: muat catatan run val keempat algoritma → periksa kelengkapan, satu embedding_id, satu env_id, dan tidak ada konfigurasi ganda → tampilkan hasil berdampingan → pilih satu konfigurasi per algoritma menurut K11 → kunci pilihan ke berkas sebelum split test dibuka.

Keluaran:
- `outputs/tuning/locked_config.json` — Kunci konfigurasi: konfigurasi terpilih per algoritma, run_id val asalnya, embedding_id, env_id, cap waktu, hash isi. Ditulis sekali.

Prasyarat: `03_exact`, `04a_hnsw`, `04b_ivf`, `04c_lsh` sudah dijalankan di instance yang sama.

Notebook ini hanya membaca catatan run; tidak memakai faiss, torch, atau vektor. Split test tidak disentuh (K2).

Keputusan: K4, K11, Kunci konfigurasi (003a), K2 (002a, 003a), K8 (004a), K9 (003a).

In [ ]:
from pathlib import Path

# Folder catatan run val dan tempat Kunci konfigurasi (K4, K9)
TUNING_DIR = Path("../outputs/tuning")
# Berkas Kunci konfigurasi, dibaca 06_final_benchmark
LOCKED_CONFIG_PATH = TUNING_DIR / "locked_config.json"

# Split yang dibaca notebook ini (K2)
SPLIT = "val"
# Jumlah run val yang diharapkan per algoritma: exact 1, efSearch 5, nprobe 7, nbits 3 (K10)
EXPECTED_RUNS = {"flat": 1, "hnsw": 5, "ivf": 7, "lsh": 3}
# Batas k-NN Recall@5 konfigurasi yang boleh dipilih menurut QPS (K11)
MIN_KNN_RECALL = 0.95

# Kolom 12 metrik K7 dan diagnostik, sesuai urutan kolom catatan run
METRIC_COLUMNS = [
    "ndcg_at_5",
    "recall_at_5",
    "mrr_at_5",
    "precision_at_5",
    "map_at_5",
    "hit_rate_at_5",
    "knn_recall_at_5",
    "relative_distance_error",
    "qps",
    "latency_p50_ms",
    "build_seconds",
    "index_size_bytes",
]
DIAGNOSTIC_COLUMNS = ["short_result_queries", "rde_excluded_terms", "rde_queries_without_terms"]

## 1. Muat dan periksa catatan run val

Angka efisiensi hanya sah dibandingkan antar-run dengan env_id yang sama (K8, 004a), dan Tetangga exact hanya sah untuk embedding_id yang sama. Kalau run val berasal dari lebih dari satu env_id atau embedding_id, notebook berhenti; memilih run mana yang dipakai belum diputuskan. Begitu juga kalau satu konfigurasi tercatat lebih dari sekali (notebook dijalankan ulang) atau jumlah run tidak sesuai K10.

In [ ]:
import json

import pandas as pd


def load_runs(tuning_dir: Path, algorithms: list[str]) -> pd.DataFrame:
    """Gabungkan catatan run semua algoritma.

    Args:
        tuning_dir: Folder outputs/tuning.
        algorithms: Nama algoritma; berkasnya `runs_<algoritma>.csv`.

    Returns:
        Semua baris run, berurutan menurut algorithms lalu urutan baris berkas.

    Raises:
        FileNotFoundError: Berkas run sebuah algoritma tidak ada.
    """
    frames = []
    for algorithm in algorithms:
        path = tuning_dir / f"runs_{algorithm}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"Catatan run tidak ditemukan: {path}")
        frames.append(pd.read_csv(path, dtype={"run_id": str, "embedding_id": str, "env_id": str}))
    return pd.concat(frames, ignore_index=True)


def validate_single_value(runs: pd.DataFrame, column: str) -> str:
    """Pastikan semua run punya nilai yang sama di satu kolom.

    Args:
        runs: Catatan run.
        column: Nama kolom, misalnya env_id atau embedding_id.

    Returns:
        Nilai tunggal kolom itu.

    Raises:
        ValueError: Ada lebih dari satu nilai.
    """
    counts = runs[column].value_counts()
    if len(counts) != 1:
        raise ValueError(
            f"Run {SPLIT} memakai {len(counts)} {column} berbeda: {counts.to_dict()}; pilihan run belum diputuskan"
        )
    return str(counts.index[0])


def validate_run_set(runs: pd.DataFrame, expected_runs: dict[str, int]) -> None:
    """Pastikan setiap konfigurasi tercatat sekali dan jumlah run per algoritma sesuai K10.

    Args:
        runs: Catatan run satu split.
        expected_runs: Jumlah run yang diharapkan per algoritma.

    Raises:
        ValueError: Ada konfigurasi ganda atau jumlah run berbeda.
    """
    is_duplicated = runs.duplicated(subset=["algorithm", "params"], keep=False)
    duplicated = runs.loc[is_duplicated, ["algorithm", "params", "run_id"]]
    if len(duplicated):
        raise ValueError(f"Konfigurasi tercatat lebih dari sekali; pilihan run ulang belum diputuskan:\n{duplicated}")
    counts = runs["algorithm"].value_counts().to_dict()
    if counts != expected_runs:
        raise ValueError(f"Jumlah run per algoritma {counts}, seharusnya {expected_runs} (K10)")


runs = load_runs(TUNING_DIR, list(EXPECTED_RUNS))
val_runs = runs.loc[runs["split"] == SPLIT].reset_index(drop=True)
embedding_id = validate_single_value(val_runs, "embedding_id")
env_id = validate_single_value(val_runs, "env_id")
validate_run_set(val_runs, EXPECTED_RUNS)
print({"runs": len(val_runs), "embedding_id": embedding_id, "env_id": env_id})

## 2. Tabel hasil val

Keempat algoritma berdampingan, satu baris per konfigurasi (K4).

In [ ]:
def to_results_table(runs: pd.DataFrame, metric_columns: list[str], diagnostic_columns: list[str]) -> pd.DataFrame:
    """Susun tabel hasil: algoritma, parameter search, metrik, dan diagnostik.

    Args:
        runs: Catatan run satu split.
        metric_columns: Kolom 12 metrik.
        diagnostic_columns: Kolom diagnostik.

    Returns:
        Tabel satu baris per konfigurasi, berurutan seperti catatan run.
    """
    params = runs["params"].map(json.loads)
    configuration = params.map(lambda value: json.dumps(value["search"] or value["build"], sort_keys=True))
    return pd.concat(
        [runs[["algorithm"]], configuration.rename("configuration"), runs[metric_columns + diagnostic_columns]],
        axis=1,
    )


results_table = to_results_table(val_runs, METRIC_COLUMNS, DIAGNOSTIC_COLUMNS)
print(results_table.to_string(index=False))

## 3. Pilih konfigurasi (K11)

Per algoritma: konfigurasi dengan QPS tertinggi di antara yang k-NN Recall@5 ≥ 0,95; kalau tidak ada yang mencapai, konfigurasi dengan k-NN Recall@5 tertinggi. Exact hanya punya satu konfigurasi. Kalau nilai tertinggi seri, baris yang lebih awal di catatan run yang terpilih.

In [ ]:
def select_config(runs: pd.DataFrame, min_knn_recall: float) -> pd.DataFrame:
    """Pilih satu konfigurasi per algoritma menurut K11.

    Args:
        runs: Catatan run val.
        min_knn_recall: Batas k-NN Recall@5.

    Returns:
        Satu baris run per algoritma ditambah kolom rule ("qps" atau "recall_fallback").
    """
    selected = []
    for _, group in runs.groupby("algorithm", sort=False):
        eligible = group.loc[group["knn_recall_at_5"] >= min_knn_recall]
        if len(eligible):
            selected.append(eligible.loc[[eligible["qps"].idxmax()]].assign(rule="qps"))
        else:
            selected.append(group.loc[[group["knn_recall_at_5"].idxmax()]].assign(rule="recall_fallback"))
    return pd.concat(selected, ignore_index=True)


selection = select_config(val_runs, MIN_KNN_RECALL)
print(selection[["algorithm", "rule", "params", "knn_recall_at_5", "qps", "run_id"]].to_string(index=False))

## 4. Kunci konfigurasi

Pilihan ditulis ke `locked_config.json` dengan cap waktu dan hash SHA-256 dari isi kanonik (semua kunci kecuali `content_hash`), sebelum split test dibuka (K2). Kunci ditulis sekali: kalau berkasnya sudah ada, pilihan baru harus sama dengan isinya dan berkas lama dibiarkan; kalau berbeda, notebook berhenti.

In [ ]:
import hashlib
from datetime import datetime, timezone


def build_locked_config(
    selection: pd.DataFrame, embedding_id: str, env_id: str, min_knn_recall: float
) -> dict[str, object]:
    """Susun isi Kunci konfigurasi tanpa hash.

    Args:
        selection: Hasil select_config.
        embedding_id: embedding_id run val.
        env_id: env_id run val.
        min_knn_recall: Batas k-NN Recall@5 K11.

    Returns:
        Isi Kunci konfigurasi dengan cap waktu.
    """
    configs = {
        row.algorithm: {
            "run_id": row.run_id,
            "params": json.loads(row.params),
            "rule": row.rule,
            "knn_recall_at_5": float(row.knn_recall_at_5),
            "qps": float(row.qps),
        }
        for row in selection.itertuples(index=False)
    }
    return {
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "embedding_id": embedding_id,
        "env_id": env_id,
        "split": SPLIT,
        "min_knn_recall": min_knn_recall,
        "configs": configs,
    }


def compute_lock_hash(content: dict[str, object]) -> str:
    """Hitung hash SHA-256 isi kanonik Kunci konfigurasi.

    Args:
        content: Isi Kunci konfigurasi; kunci content_hash diabaikan.

    Returns:
        Hash heksadesimal 64 karakter.
    """
    body = {key: value for key, value in content.items() if key != "content_hash"}
    canonical = json.dumps(body, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def save_locked_config(content: dict[str, object], path: Path) -> bool:
    """Tulis Kunci konfigurasi sekali.

    Args:
        content: Isi Kunci konfigurasi beserta content_hash.
        path: Berkas tujuan.

    Returns:
        True kalau berkas baru ditulis, False kalau berkas lama berisi pilihan yang sama dan dibiarkan.

    Raises:
        ValueError: Berkas lama berisi pilihan berbeda.
    """
    if path.is_file():
        with path.open(encoding="utf-8") as file:
            previous = json.load(file)
        keys = ["embedding_id", "env_id", "split", "min_knn_recall", "configs"]
        if any(previous.get(key) != content[key] for key in keys):
            raise ValueError(f"{path} sudah ada dengan pilihan berbeda; Kunci konfigurasi tidak ditimpa")
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)
    return True


locked_config = build_locked_config(selection, embedding_id, env_id, MIN_KNN_RECALL)
locked_config["content_hash"] = compute_lock_hash(locked_config)
written = save_locked_config(locked_config, LOCKED_CONFIG_PATH)
print({"path": str(LOCKED_CONFIG_PATH), "written": written, "content_hash": locked_config["content_hash"]})